# Debug knee exo — AB03_Ilseung RA 0.8 m/s

Single-trial debugger for `ab03_ilseung_knee_0p8mps_ra_exo_on`.

Uses **logged telemetry** from the trial NPZ (no model re-inference):
1. **GPIO sync** — falling-edge alignment (telemetry vs mocap jet)
2. **GT vs model** — Vicon ID/mass − `cmd_R`/mass vs logged `model_out_nmpkg_raw`

GT net moment is zero-phase 6 Hz LPF; model output is the raw logged N·m/kg trace.


In [ ]:
import io
from pathlib import Path
from typing import Dict, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import butter, sosfilt, sosfiltfilt

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
TELEMETRY_ROOT = PROJECT_ROOT

TRIAL_STEM = 'ab04_changseob_knee_0p8mps_rd_exo_on'
SUBJECT = 'AB04_Changseob'
COND, SPEED = 'RD', '0p8mps'

EXO_KIND = 'knee-exo'
MOCAP_FS_HZ = 1000.0
MOMENT_COL = 'knee_angle_r_moment'
LPF_CUTOFF_HZ, LPF_ORDER = 6.0, 4
GT_LPF_MODE = 'zero_phase'

SUBJECT_TOKEN_TO_DIR = {
    'ab01_jinwoo': 'AB01_Jinwoo', 'ab02_oscar': 'AB02_Oscar', 'ab03_ilseung': 'AB03_Ilseung',
    'ab04_changseob': 'AB04_Changseob', 'ab05_maria': 'AB05_Maria', 'ab06_jimin': 'AB06_Jimin',
    'ab07_amy': 'AB07_Amy', 'ab08_seokhyun': 'AB08_Seokhyun',
}
SUBJECT_MASS_KG = {
    'ab01_jinwoo': 88.0, 'ab02_oscar': 71.1, 'ab03_ilseung': 84.4, 'ab04_changseob': 74.0,
    'ab05_maria': 55.0, 'ab06_jimin': 82.6, 'ab07_amy': 51.3, 'ab08_seokhyun': 71.9,
}

GPIO_PALETTE = {'mocap': '#90A4AE', 'telemetry': '#FF9800'}

print(f'Trial: {TRIAL_STEM}')
print(f'Subject/cond: {SUBJECT} {COND}_{SPEED}')
print(f'Processed root: {PROCESSED_ROOT}')


Trial: ab05_maria_knee_0p8mps_rd_exo_on
Subject/cond: AB05_Maria RD_0p8mps
Processed root: /media/metamobility3/Samsung_T52/Results/processed


In [26]:
def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    arr = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * float(fs_hz)
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(arr) < 4:
        return arr
    sos = butter(int(order), float(cutoff_hz) / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, arr) if mode == 'zero_phase' else sosfilt(sos, arr)


def lpf_nan(x, fs_hz, cutoff_hz, order, mode):
    arr = np.asarray(x, dtype=np.float64).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(3, order + 1):
        return arr
    if finite.all():
        return butter_lpf(arr, fs_hz, cutoff_hz, order, mode)
    idx = np.arange(arr.size, dtype=np.float64)
    filled = np.interp(idx, idx[finite], arr[finite])
    out = butter_lpf(filled, fs_hz, cutoff_hz, order, mode)
    out[~finite] = np.nan
    return out


def rmse_r2(y_true, y_pred):
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    return rmse, float(1.0 - ss_res / (ss_tot + 1e-12))


def read_sto(path: Path):
    with open(path) as f:
        lines = f.readlines()
    end_idx = next(i for i, l in enumerate(lines) if l.strip().lower() == 'endheader')
    cols = lines[end_idx + 1].strip().split()
    data = np.loadtxt(io.StringIO(''.join(lines[end_idx + 2:])))
    if data.ndim == 1:
        data = data.reshape(1, -1)
    return cols, data


def extract_model_out_nmpkg_raw(npz) -> Tuple[np.ndarray, str]:
    if 'model_out_nmpkg_raw' in npz.files:
        return np.asarray(npz['model_out_nmpkg_raw'], dtype=np.float64), 'model_out_nmpkg_raw'
    if 'model_out_nmpkg' in npz.files:
        return np.asarray(npz['model_out_nmpkg'], dtype=np.float64), 'model_out_nmpkg'
    raise KeyError(f'No model output key; available: {sorted(npz.files)}')


def extract_applied_cmd_nm(npz) -> Tuple[np.ndarray, str]:
    for k in ('cmd_R', 'cmd_L'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError(f'No cmd_R/cmd_L; available: {sorted(npz.files)}')


def infer_fs_hz(time_s, default_fs=100.0):
    if time_s is None or len(time_s) < 3:
        return float(default_fs)
    dt = np.diff(np.asarray(time_s, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    return float(1.0 / np.median(dt)) if dt.size else float(default_fs)


def _subject_token(stem: str) -> str:
    return '_'.join(stem.lower().split('_')[:2])


def subject_dir_from_stem(stem: str) -> Path:
    token = _subject_token(stem)
    name = SUBJECT_TOKEN_TO_DIR[token]
    p = PROCESSED_ROOT / name
    if not p.is_dir():
        raise FileNotFoundError(p)
    return p


def trial_cond_speed(stem: str) -> Tuple[str, str]:
    parts = stem.lower().split('_')
    return parts[4].upper(), parts[3]


def parse_mocap_csv(path: Path, fs: float = MOCAP_FS_HZ):
    df = pd.read_csv(path, skiprows=[0, 1, 2, 4], header=0, low_memory=False, on_bad_lines='skip')
    df = df[pd.to_numeric(df['Frame'], errors='coerce').notna()].copy()
    df['jet'] = pd.to_numeric(df['jet'], errors='coerce')
    df = df.dropna(subset=['jet']).reset_index(drop=True)
    return np.arange(len(df)) / fs, df['jet'].to_numpy(dtype=float)


def normalize_gpio(gpio: np.ndarray) -> np.ndarray:
    arr = np.asarray(gpio, dtype=np.float64)
    g_range = arr.max() - arr.min()
    return arr if g_range <= 0 else (arr - arr.min()) / g_range


def first_falling_edge(signal: np.ndarray, threshold: float = 0.5) -> Optional[int]:
    above = np.asarray(signal, dtype=np.float64) > threshold
    for i in range(1, len(above)):
        if above[i - 1] and not above[i]:
            return i
    return None


def extract_gpio(npz) -> Tuple[np.ndarray, str]:
    for k in ('gpio_output', 'GPIO', 'gpio', 'trigger'):
        if k in npz.files:
            return np.asarray(npz[k], dtype=np.float64), k
    raise KeyError('No GPIO key in npz')


def gpio_offset_s(t_exo, g_exo, t_mocap, g_mocap):
    idx_exo = first_falling_edge(g_exo)
    idx_mocap = first_falling_edge(normalize_gpio(g_mocap))
    if idx_exo is None or idx_mocap is None:
        return None, idx_exo, idx_mocap
    return float(t_mocap[idx_mocap] - t_exo[idx_exo]), idx_exo, idx_mocap


def resolve_trial_paths(trial_stem: str) -> Dict[str, Path]:
    cond, speed = trial_cond_speed(trial_stem)
    subj_dir = subject_dir_from_stem(trial_stem)
    return {
        'npz': TELEMETRY_ROOT / f'{trial_stem}.npz',
        'mocap': subj_dir / EXO_KIND / 'mocap' / f'{cond}_{speed}.csv',
        'id': subj_dir / EXO_KIND / 'id' / f'{cond}_{speed}_id.sto',
        'cond': cond,
        'speed': speed,
        'subject_dir': subj_dir,
    }


def load_gpio_sync_data(trial_stem: str) -> Dict:
    paths = resolve_trial_paths(trial_stem)
    for key in ('npz', 'mocap', 'id'):
        if not paths[key].exists():
            raise FileNotFoundError(f'Missing {key}: {paths[key]}')

    d = np.load(str(paths['npz']), allow_pickle=True)
    gpio, gpio_key = extract_gpio(d)
    t_raw = np.asarray(d['time'], dtype=np.float64) if 'time' in d.files else np.arange(len(gpio), dtype=np.float64)
    n = min(len(t_raw), len(gpio))
    t_raw, gpio = t_raw[:n], gpio[:n]

    t_mocap, gpio_mocap = parse_mocap_csv(paths['mocap'])
    offset_s, idx_exo, idx_mocap = gpio_offset_s(t_raw, gpio, t_mocap, gpio_mocap)

    return {
        'trial': trial_stem,
        'sync_method': 'gpio_falling_edge',
        'paths': paths,
        'npz': d,
        'gpio_key': gpio_key,
        'offset_s': offset_s,
        'idx_exo': idx_exo,
        'idx_mocap': idx_mocap,
        'fs_npz_hz': infer_fs_hz(t_raw),
        'fs_mocap_hz': infer_fs_hz(t_mocap, default_fs=MOCAP_FS_HZ),
        't_npz': t_raw,
        'gpio_npz': gpio,
        't_mocap': t_mocap,
        'gpio_mocap_raw': gpio_mocap,
        'gpio_mocap_norm': normalize_gpio(gpio_mocap),
        't_npz_aligned': t_raw + float(offset_s) if offset_s is not None else t_raw.copy(),
    }


def load_moment_waveforms(trial_stem: str, sync: Dict) -> Dict:
    """GPIO-synced GT vs logged model_out_nmpkg_raw from telemetry NPZ."""
    paths = sync['paths']
    mass = SUBJECT_MASS_KG[_subject_token(trial_stem)]
    d = sync['npz']

    applied_nm, applied_key = extract_applied_cmd_nm(d)
    model_out_raw, model_out_key = extract_model_out_nmpkg_raw(d)

    n = min(len(sync['t_npz']), len(applied_nm), len(model_out_raw))
    t_aligned = sync['t_npz_aligned'][:n].astype(np.float64)
    applied_nm = applied_nm[:n]
    model_out_nmpkg_raw = np.asarray(model_out_raw[:n], dtype=np.float64)
    fs_hz = infer_fs_hz(sync['t_npz'][:n])

    cols, id_data = read_sto(paths['id'])
    t_id = id_data[:, cols.index('time')]
    id_moment_nm = id_data[:, cols.index(MOMENT_COL)]

    id_nm_raw = np.interp(t_aligned, t_id, id_moment_nm, left=np.nan, right=np.nan)
    id_nmpkg_raw = id_nm_raw / mass
    applied_nmpkg_raw = applied_nm / mass

    net_raw_nmpkg = id_nmpkg_raw - applied_nmpkg_raw
    gt_nmpkg = lpf_nan(net_raw_nmpkg, fs_hz, LPF_CUTOFF_HZ, LPF_ORDER, GT_LPF_MODE)

    return {
        'trial': trial_stem,
        't': t_aligned,
        'gt_nmpkg': gt_nmpkg,
        'model_out_nmpkg_raw': model_out_nmpkg_raw,
        'applied_key': applied_key,
        'model_out_key': model_out_key,
        'mass_kg': mass,
        'fs_hz': fs_hz,
    }


print('Helpers ready.')


Helpers ready.


## 1. File + NPZ inventory

In [27]:
PATHS = resolve_trial_paths(TRIAL_STEM)
for name, p in PATHS.items():
    if isinstance(p, Path):
        status = 'OK' if p.exists() else 'MISSING'
        print(f'[{status}] {name}: {p}')

d = np.load(str(PATHS['npz']), allow_pickle=True)
rows = []
for k in sorted(d.files):
    v = d[k]
    if hasattr(v, 'shape'):
        arr = np.asarray(v, dtype=np.float64)
        rows.append({
            'key': k,
            'n': len(arr),
            'min': float(np.nanmin(arr)),
            'max': float(np.nanmax(arr)),
            'nan_pct': float(np.mean(~np.isfinite(arr)) * 100),
        })
display(pd.DataFrame(rows))

[OK] npz: /home/metamobility3/Jinwoo/os_kinetics/ab05_maria_knee_0p8mps_rd_exo_on.npz
[OK] mocap: /media/metamobility3/Samsung_T52/Results/processed/AB05_Maria/knee-exo/mocap/RD_0p8mps.csv
[OK] id: /media/metamobility3/Samsung_T52/Results/processed/AB05_Maria/knee-exo/id/RD_0p8mps_id.sto
[OK] subject_dir: /media/metamobility3/Samsung_T52/Results/processed/AB05_Maria


,key,n,min,max,nan_pct
0,GPIO,8000,0.000000,1.000000,0.0
1,K_l,8000,0.000000,0.000000,0.0
2,K_r,8000,0.000000,0.000000,0.0
3,Soft_ctrl_l,8000,0.000000,0.000000,0.0
4,Soft_ctrl_r,8000,0.000000,0.000000,0.0
5,cmd_L,8000,0.000000,0.000000,0.0
6,cmd_R,8000,-0.834558,5.204569,0.0
7,gyro_shank_r,8000,-3.959450,6.781775,0.0
8,gyro_thigh_r,8000,-1.983184,2.156652,0.0
9,knee_angle_l,8000,0.000000,0.000000,0.0


## 2. GPIO sync

In [28]:
SYNC = load_gpio_sync_data(TRIAL_STEM)
if SYNC['offset_s'] is None:
    raise RuntimeError('No GPIO falling edge found')

print(f"GPIO key: {SYNC['gpio_key']}")
print(f"Offset: {SYNC['offset_s']:+.4f} s")
print(f"Telemetry edge idx/time: {SYNC['idx_exo']} @ {SYNC['t_npz'][SYNC['idx_exo']]:.4f} s")
print(f"Mocap edge idx/time:     {SYNC['idx_mocap']} @ {SYNC['t_mocap'][SYNC['idx_mocap']]:.4f} s")
print(f"Sample rates: telemetry={SYNC['fs_npz_hz']:.2f} Hz, mocap={SYNC['fs_mocap_hz']:.2f} Hz")

WAVE = load_moment_waveforms(TRIAL_STEM, SYNC)
print(f"\nApplied key: {WAVE['applied_key']} | Model key: {WAVE['model_out_key']}")
print(f"Duration: {WAVE['t'][-1]-WAVE['t'][0]:.1f} s | n={len(WAVE['t'])}")


GPIO key: GPIO
Offset: -0.1361 s
Telemetry edge idx/time: 220 @ 2.2101 s
Mocap edge idx/time:     2074 @ 2.0740 s
Sample rates: telemetry=100.00 Hz, mocap=1000.00 Hz

Applied key: cmd_R | Model key: model_out_nmpkg_raw
Duration: 80.0 s | n=8000


In [29]:
def draw_gpio_sync(sync: Dict, window_s: float = 4.0) -> None:
    t_edge_mocap = float(sync['t_mocap'][sync['idx_mocap']])
    t_edge_npz = float(sync['t_npz'][sync['idx_exo']])
    t0, t1 = t_edge_mocap - 0.5, t_edge_mocap + window_s
    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

    m_npz = (sync['t_npz'] >= t_edge_npz - 0.5) & (sync['t_npz'] <= t_edge_npz + window_s)
    m_mocap = (sync['t_mocap'] >= t0) & (sync['t_mocap'] <= t1)
    axs[0].plot(sync['t_npz'][m_npz], sync['gpio_npz'][m_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label=f"Telemetry ({sync['gpio_key']})")
    axs[0].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[0].axvline(t_edge_npz, color=GPIO_PALETTE['telemetry'], ls=':')
    axs[0].axvline(t_edge_mocap, color=GPIO_PALETTE['mocap'], ls=':')
    axs[0].set_ylabel('Amplitude (a.u.)')
    axs[0].set_title('Before sync')
    axs[0].legend(); axs[0].grid(alpha=0.25)

    m_a_npz = (sync['t_npz_aligned'] >= t0) & (sync['t_npz_aligned'] <= t1)
    axs[1].plot(sync['t_mocap'][m_mocap], sync['gpio_mocap_norm'][m_mocap], color=GPIO_PALETTE['mocap'], lw=1.2, label='Mocap jet (norm)')
    axs[1].plot(sync['t_npz_aligned'][m_a_npz], sync['gpio_npz'][m_a_npz], color=GPIO_PALETTE['telemetry'], lw=1.4, ls='--', label='Telemetry shifted')
    axs[1].axvline(t_edge_mocap, color='black', ls=':')
    axs[1].set_xlabel('Mocap time (s)'); axs[1].set_ylabel('Amplitude (a.u.)')
    axs[1].set_title(f'After sync | offset {sync["offset_s"]:+.4f} s')
    axs[1].legend(); axs[1].grid(alpha=0.25)
    fig.suptitle(f"{sync['trial']} | GPIO sync", y=1.01)
    fig.tight_layout(); plt.show()

gpio_window = widgets.FloatSlider(value=4.0, min=1.0, max=15.0, step=0.5, description='Window (s):')
gpio_out = widgets.Output()

def _draw_gpio(*_):
    with gpio_out:
        gpio_out.clear_output(wait=True)
        draw_gpio_sync(SYNC, gpio_window.value)

gpio_window.observe(_draw_gpio, names='value')
display(widgets.VBox([gpio_window, gpio_out]))
_draw_gpio()

## 3. GT vs model_out_nmpkg_raw + residual

GT = Vicon ID/mass − cmd_R/mass (zero-phase 6 Hz LPF); model = logged `model_out_nmpkg_raw` (no extra filtering).


In [ ]:
rmse, r2 = rmse_r2(WAVE['gt_nmpkg'], WAVE['model_out_nmpkg_raw'])
print(f'GT vs model_out_nmpkg_raw: RMSE={rmse:.4f} N·m/kg | R²={r2:.4f}')

res_out = widgets.Output()
res_slider = widgets.FloatRangeSlider(description='Time (s):', continuous_update=False, readout_format='.2f', layout=widgets.Layout(width='760px'))


def draw_gt_residual(wave: Dict, t_window: Tuple[float, float]) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    t0, t1 = t_window
    gt = wave['gt_nmpkg']
    model = wave['model_out_nmpkg_raw']
    m = (t_rel >= t0) & (t_rel <= t1) & np.isfinite(gt) & np.isfinite(model)

    fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
    axs[0].plot(t_rel[m], gt[m], color='#1e88e5', lw=2.0, label='GT (ID/mass − cmd_R/mass)')
    axs[0].plot(t_rel[m], model[m], color='#e53935', lw=1.6, ls='--', label='model_out_nmpkg_raw (logged)')
    axs[0].set_ylabel('N·m/kg')
    axs[0].set_title(f"RMSE={rmse:.4f} | R²={r2:.4f}")
    axs[0].legend(); axs[0].grid(alpha=0.25)

    axs[1].plot(t_rel[m], (model - gt)[m], color='#9c27b0', lw=1.4, label='model_out_nmpkg_raw − GT')
    axs[1].axhline(0, color='black', ls=':')
    axs[1].set_ylabel('Residual (N·m/kg)'); axs[1].set_xlabel('Time (s)')
    axs[1].legend(); axs[1].grid(alpha=0.25)
    fig.tight_layout()
    with res_out:
        res_out.clear_output(wait=True); plt.show()


def _init_res_slider(wave: Dict) -> None:
    t_rel = wave['t'] - np.nanmin(wave['t'])
    res_slider.min = float(t_rel[0]); res_slider.max = float(t_rel[-1])
    res_slider.step = max((res_slider.max - res_slider.min) / 500, 1e-3)
    res_slider.value = (res_slider.min, res_slider.max)


def _redraw_res(*_):
    draw_gt_residual(WAVE, res_slider.value)

res_slider.observe(_redraw_res, names='value')
_init_res_slider(WAVE)
display(widgets.VBox([res_slider, res_out]))
_redraw_res()


GT vs model_out_nmpkg_raw: RMSE=0.9082 N·m/kg | R²=-0.2426
